# Orbit Wars — Colab Training Pipeline (v2 OrbitNet, A100)

GPU-accelerated training on Colab with Google Drive persistence.

This notebook runs the **v2 pipeline** with the four structural fixes and the
**Expert Iteration (ExIt)** pivot identified in `rl_research/REPORT.md`:

- **Fix 1 — dedicated masked ship-fraction head** (factored `(target, fraction)`;
  BC clones fleet sizes; PPO/ExIt get a real gradient on quantity).
- **Fix 2 — mean per-planet entropy** (stable exploration pressure).
- **Fix 3 — symlog value targets** (`ppo.value_symlog`, scale-robust value).
- **Fix 4 — longer rollouts** (episodes complete → terminal signal reaches updates).

Two pipelines (set `PIPELINE` in the config cell):
1. **`exit`** (recommended) — BC-clone apex → per-planet lookahead **search against the
   ground-truth simulator** → supervised distillation. Planning, not policy-gradient.
2. **`ppo`** — v2 PPO with all four fixes + parallel CPU rollout collection.

### A100 efficiency note
The OrbitNet is tiny (~0.55M params), so the **GPU is never the compute bottleneck** —
game simulation (collection) and lookahead (search) are **CPU-bound**. We therefore:
- enable **TF32** and use **large batches** so the A100 chews through BC / distillation /
  PPO updates,
- fan **search across all vCPUs** (`exit.search_workers`) and run **parallel rollout
  workers** (`ppo.num_workers`) for the PPO path,
- **cache apex demos on Drive** so the one-time collection isn't repeated across runs.

In [ ]:
#@title 1. Setup: Mount Drive, Clone Repo, Install Dependencies

import os, sys

from google.colab import drive
drive.mount('/content/drive')

DRIVE_OUTPUT = '/content/drive/MyDrive/orbit_wars_outputs'
for sub in ['', '/checkpoints', '/logs', '/submissions']:
    os.makedirs(DRIVE_OUTPUT + sub, exist_ok=True)
print(f'Drive output dir: {DRIVE_OUTPUT}')

REPO_URL = 'https://github.com/oshbocker/orbit_wars.git'  # <-- EDIT if needed
REPO_DIR = '/content/orbit_wars'
if os.path.exists(REPO_DIR):
    print('Repo present, pulling latest...')
    os.system(f'cd {REPO_DIR} && git pull')
else:
    os.system(f'git clone {REPO_URL} {REPO_DIR}')

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print(f'Working dir: {os.getcwd()}')

# v2 needs only kaggle-environments + the usual scientific stack (no SB3).
os.system('pip install -q --upgrade "kaggle-environments>=1.28.0" pyyaml tensorboard')
print('\nSetup complete.')

In [ ]:
#@title 2. GPU Verification + A100 performance flags

import os, torch

print(f'PyTorch {torch.__version__}')
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    print(f'GPU: {name}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    # A100 / Ampere: enable TF32 for fast fp32 matmul (free speedup).
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    DEVICE = 'cuda'
else:
    print('WARNING: no GPU. Runtime > Change runtime type > GPU (A100).')
    DEVICE = 'cpu'

CPU_COUNT = os.cpu_count() or 8
print(f'\nDevice: {DEVICE} | vCPUs: {CPU_COUNT}')
print('Note: collection/search are CPU-bound; the GPU accelerates neural training.')

## Configuration

Pick the pipeline and apply A100-tuned overrides. The base YAMLs
(`configs/v2_exit.yaml`, `configs/v2_bc.yaml`) already carry the four fixes;
here we scale batch sizes for the GPU and parallelism for the vCPUs, and point
all outputs (incl. the cached apex demo set) at Google Drive.

In [ ]:
#@title 3. Experiment Config

from v2.config import load_v2_config

PIPELINE = 'exit'  #@param ['exit', 'ppo']

base_yaml = 'configs/v2_exit.yaml' if PIPELINE == 'exit' else 'configs/v2_bc.yaml'
cfg = load_v2_config(base_yaml)

# ── Common: device + Drive persistence + demo cache ─────────────────────────
cfg.device = 'cuda'
cfg.save_dir = f'{DRIVE_OUTPUT}/checkpoints'
cfg.log_dir = f'{DRIVE_OUTPUT}/logs'
cfg.imitation.bc_cache_path = f'{DRIVE_OUTPUT}/demos_apex_v2.pkl'  # v2 = higher-fidelity mapping; collect once, reuse

if PIPELINE == 'exit':
    cfg.run_name = 'v2_exit_a100'
    # BC warm start: collection is CPU-bound (cached); epochs are GPU-cheap.
    cfg.imitation.enabled = True
    cfg.imitation.bc_games = 300
    cfg.imitation.bc_epochs = 60
    cfg.imitation.bc_batch_size = 1024
    # ExIt: search is the cost -> fan across vCPUs; GPU does big-batch distillation.
    cfg.exit.iterations = 60
    cfg.exit.games_per_iter = 12
    cfg.exit.search_depth = 16
    cfg.exit.search_candidates = 12
    cfg.exit.train_epochs = 6
    cfg.exit.train_batch_size = 2048
    cfg.exit.dataset_max_iters = 4
    cfg.exit.search_workers = CPU_COUNT
    cfg.checkpoint_every = 5
    cfg.eval.eval_every = 5
    cfg.eval.eval_games = 20
else:  # ppo with all four fixes
    cfg.run_name = 'v2_ppo_a100'
    cfg.imitation.enabled = True
    cfg.imitation.bc_games = 300
    cfg.imitation.bc_epochs = 60
    cfg.ppo.num_workers = CPU_COUNT     # parallel CPU rollout collection (key lever)
    cfg.ppo.num_envs = 8
    cfg.ppo.rollout_steps = 128         # Fix 4: episodes complete
    cfg.ppo.total_updates = 5000
    cfg.ppo.minibatch_size = 2048       # large batch for A100 throughput
    cfg.ppo.epochs = 4
    cfg.ppo.gamma = 0.997               # long-horizon (meaningful with symlog)
    cfg.ppo.value_symlog = True         # Fix 3
    cfg.ppo.ent_coef = 0.01
    cfg.ppo.ent_coef_end = 0.0          # entropy anneal (Fix 2 is in the code)
    cfg.eval.eval_every = 250
    cfg.eval.eval_games = 20

print(f'PIPELINE      : {PIPELINE}')
print(f'run_name      : {cfg.run_name}')
print(f'device        : {cfg.device}')
print(f'model         : embed={cfg.model.embed_dim} layers={cfg.model.n_layers} '
      f'n_fractions={cfg.model.n_fractions}')
print(f'ship_fractions: {cfg.env.ship_fractions}')
print(f'BC            : enabled={cfg.imitation.enabled} games={cfg.imitation.bc_games} '
      f'epochs={cfg.imitation.bc_epochs} cache={cfg.imitation.bc_cache_path}')
if PIPELINE == 'exit':
    print(f'ExIt          : iters={cfg.exit.iterations} games/iter={cfg.exit.games_per_iter} '
          f'depth={cfg.exit.search_depth} cand={cfg.exit.search_candidates} '
          f'workers={cfg.exit.search_workers} batch={cfg.exit.train_batch_size}')
else:
    print(f'PPO           : updates={cfg.ppo.total_updates} envs={cfg.ppo.num_envs} '
          f'workers={cfg.ppo.num_workers} rollout={cfg.ppo.rollout_steps} '
          f'minibatch={cfg.ppo.minibatch_size} gamma={cfg.ppo.gamma} '
          f'value_symlog={cfg.ppo.value_symlog} ent={cfg.ppo.ent_coef}->{cfg.ppo.ent_coef_end}')

In [ ]:
#@title 4. Train

import yaml
from v2.config import v2_config_to_dict

temp_cfg = '/tmp/v2_train_cfg.yaml'
with open(temp_cfg, 'w') as f:
    yaml.dump(v2_config_to_dict(cfg), f, sort_keys=True)
print(f'Wrote config -> {temp_cfg}')

module = 'v2.exit_train' if PIPELINE == 'exit' else 'v2.train'
print(f'Launching: python -m {module} --config {temp_cfg}\n')
!python -m {module} --config {temp_cfg}

CHECKPOINT_DIR = f'{cfg.save_dir}/{cfg.run_name}'
print(f'\nCheckpoints saved to: {CHECKPOINT_DIR}')

## Submission

Saves a reliable **apex** baseline plus a **v2 trained-model bundle** (`main.py` =
`v2/agent.py` + checkpoint + the `v2`/`src` packages it imports). For Kaggle
simulation comps, upload the bundle directory; `agent.py` auto-loads
`ckpt_last.pt` from `/kaggle_simulations/agent/`.

In [ ]:
#@title 5. Generate Submission

import shutil
from pathlib import Path

sub_dir = Path(DRIVE_OUTPUT) / 'submissions'
sub_dir.mkdir(parents=True, exist_ok=True)

# Reliable apex baseline
try:
    from agents.rl_agent import export_submission
    export_submission(None, output_path=str(sub_dir / 'submission_apex.py'), mode='apex')
    print(f'Apex baseline: {sub_dir / "submission_apex.py"}')
except Exception as e:
    print(f'(apex export skipped: {e})')

# v2 trained-model bundle
ckpt_src = Path(CHECKPOINT_DIR) / 'ckpt_last.pt'
bundle = sub_dir / f'{cfg.run_name}_bundle'
if ckpt_src.exists():
    bundle.mkdir(parents=True, exist_ok=True)
    shutil.copy2('v2/agent.py', bundle / 'main.py')
    shutil.copy2(ckpt_src, bundle / 'ckpt_last.pt')
    shutil.copytree('v2', bundle / 'v2', dirs_exist_ok=True)
    shutil.copytree('src', bundle / 'src', dirs_exist_ok=True)
    print(f'v2 bundle: {bundle}  (main.py + ckpt_last.pt + v2/ + src/)')
else:
    print(f'No checkpoint at {ckpt_src} yet.')

print('\nUpload examples:')
print(f'  kaggle competitions submit orbit-wars -f "{sub_dir/"submission_apex.py"}" -m "apex"')
print(f'  # v2: zip the bundle dir and submit per the comp\'s directory-submission docs')

## Evaluation

Run the trained OrbitNet head-to-head vs apex and random.

In [ ]:
#@title 6. Evaluate Trained Model

import torch
from pathlib import Path
from v2.model import OrbitNet
from v2.train import make_v2_eval_agent
from evaluation.evaluate import run_games, print_results
from agents.apex import agent as apex_agent
from kaggle_environments.envs.orbit_wars.orbit_wars import random_agent

ckpt_path = Path(CHECKPOINT_DIR) / 'ckpt_last.pt'
if not ckpt_path.exists():
    print(f'No checkpoint at {ckpt_path}. Available:')
    for p in sorted(Path(DRIVE_OUTPUT, 'checkpoints').rglob('ckpt_last.pt')):
        print(f'  {p}')
else:
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = OrbitNet(cfg.model).to(device)
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=True)
    model.load_state_dict(ckpt['model'])
    model.eval()
    print(f'Loaded {ckpt_path} (iter/update {ckpt.get("update", "?")})')
    eval_agent = make_v2_eval_agent(model, cfg, device)

    N = 20
    print(f'\nRL vs Apex ({N} games):')
    print_results('v2', 'apex', run_games(eval_agent, apex_agent, n_games=N))
    print(f'RL vs Random ({N} games):')
    print_results('v2', 'random', run_games(eval_agent, random_agent, n_games=N))

## Monitoring

TensorBoard shows train/*, eval/*, and bc/* metrics.
**Run this last** — TensorBoard can block cells below it.

In [ ]:
#@title 7. TensorBoard

%load_ext tensorboard
%tensorboard --logdir {DRIVE_OUTPUT}/logs